pull data

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import pyodbc
# -----------------------------------------------------------------------------
# ANALYSIS PERIOD
# Summer 2026
# -----------------------------------------------------------------------------

analysis_start_date = pd.Timestamp("2026-03-29")
analysis_end_date = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)

arrival_dates = pd.date_range(
    analysis_start_date,
    analysis_end_date,
    freq="D"
)

print(f"Analysis period:")
print(f"Start: {analysis_start_date.date()}")
print(f"End:   {analysis_end_date.date()}")

# -----------------------------------------------------------------------------
# FORECAST FILE LOCATION
# -----------------------------------------------------------------------------

user = 'jamie_douglas'

dsn = 'AzureConnection'
# user = 'christina_k'

username = '{user}@edinburghairport.com'

connect = pyodbc.connect('DSN=' + dsn + ';')

forecast_folder = Path(
    rf"C:\Users\{user}\Edinburgh Airport Limited\Business Planning - Documents\02. Capacity Areas\09. Car Parking\01. FastPark\FastPark Model Rebuild 2025\FastPark Short-Term Model Rebuild\Daily Forecast Output"
)

# -----------------------------------------------------------------------------
# LOAD ACTUALS
# -----------------------------------------------------------------------------

# Assumes:
# - connect already exists
# - pyodbc/sqlalchemy connection already created

# We need actuals for the last completed week only

sql = f"""
SELECT *
FROM FastPark.v_EntryAndExits
WHERE [CheckInStarted] >= '{analysis_start_date.strftime('%Y-%m-%d')}'
    AND [CheckInStarted] < '{(analysis_end_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')}'
"""

fastpark_actual_entries_df = pd.read_sql(sql, connect)

fastpark_actual_entries_df = fastpark_actual_entries_df.drop_duplicates(subset='BookingReference')


sql = f"""
SELECT *
FROM FastPark.v_EntryAndExits
WHERE [ActualCheckedOutDate] >= '{analysis_start_date.strftime('%Y-%m-%d')}'
    AND [ActualCheckedOutDate] < '{(analysis_end_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')}'
"""

fastpark_actual_exits_df = pd.read_sql(sql, connect)

fastpark_actual_exits_df = fastpark_actual_exits_df.drop_duplicates(subset='BookingReference')

# -----------------------------------------------------------------------------
# PREP ACTUALS
# -----------------------------------------------------------------------------

# Entries
fastpark_actual_entries_df["Date"] = pd.to_datetime(
    fastpark_actual_entries_df["CheckInStarted"]
).dt.date

actual_entries_daily = (
    fastpark_actual_entries_df
    .groupby("Date")
    .size()
    .reset_index(name="ActualEntries")
)

# Exits
fastpark_actual_exits_df["Date"] = pd.to_datetime(
    fastpark_actual_exits_df["ActualCheckedOutDate"]
).dt.date

actual_exits_daily = (
    fastpark_actual_exits_df
    .groupby("Date")
    .size()
    .reset_index(name="ActualExits")
)

# Merge actuals
actuals_daily = pd.merge(
    actual_entries_daily,
    actual_exits_daily,
    on="Date",
    how="outer"
).fillna(0)

actuals_daily["Date"] = pd.to_datetime(actuals_daily["Date"])

print("\nActuals daily:")
print(actuals_daily.head())

# -----------------------------------------------------------------------------
# LEAD TIMES TO CHECK
# -----------------------------------------------------------------------------

lead_times = [42, 35, 28, 21, 14, 7]

# Results dictionary
entries_accuracy_results = {}
exits_accuracy_results = {}
# Results dictionary
entries_accuracy_results = {}
exits_accuracy_results = {}

# NEW
entries_forecast_results = {}
exits_forecast_results = {}

entries_actual_results = {}
exits_actual_results = {}
# -----------------------------------------------------------------------------
# LOOP THROUGH EACH DAY IN LAST WEEK
# -----------------------------------------------------------------------------

for target_date in arrival_dates:

    target_date_only = target_date.date()

    # Get actuals
    actual_row = actuals_daily[
        actuals_daily["Date"].dt.date == target_date_only
    ]

    if actual_row.empty:
        print(f"\nNo actuals found for {target_date_only}")
        continue

    actual_entries = actual_row["ActualEntries"].iloc[0]
    actual_exits = actual_row["ActualExits"].iloc[0]

    entries_accuracy_results[target_date_only] = {}
    exits_accuracy_results[target_date_only] = {}
    
    entries_forecast_results[target_date_only] = {}
    exits_forecast_results[target_date_only] = {}

    entries_actual_results[target_date_only] = {}
    exits_actual_results[target_date_only] = {}
    # -------------------------------------------------------------------------
    # CHECK EACH LEAD TIME
    # -------------------------------------------------------------------------

    for lead in lead_times:

        # Forecast file date
        forecast_run_date = target_date - pd.Timedelta(days=lead)

        if forecast_run_date.strftime('%Y-%m-%d') == '2026-06-22':
            forecast_filename = (
                f"{forecast_run_date.strftime('%Y-%m-%d')}_daily_forecast - Copy.csv"
            )
        else:
            forecast_filename = (
                f"{forecast_run_date.strftime('%Y-%m-%d')}_daily_forecast.csv"
            )

        forecast_path = forecast_folder / forecast_filename

        print(f"\nChecking forecast file:")
        print(f"Target date: {target_date_only}")
        print(f"Lead time: {lead}")
        print(f"Forecast file: {forecast_filename}")

        # ---------------------------------------------------------------------
        # FILE EXISTS?
        # ---------------------------------------------------------------------

        if not forecast_path.exists():
            print("Forecast file NOT found - skipping")
            entries_accuracy_results[target_date_only][lead] = np.nan
            exits_accuracy_results[target_date_only][lead] = np.nan
            continue

        try:

            # -----------------------------------------------------------------
            # LOAD FORECAST FILE
            # -----------------------------------------------------------------
            # -----------------------------------------------------------------
            # LOAD FORECAST FILE
            # -----------------------------------------------------------------

            forecast_df = pd.read_csv(forecast_path)

            print(f"Loaded rows: {len(forecast_df)}")

            # -----------------------------------------------------------------
            # HANDLE POSSIBLE MISSING / DIFFERENT COLUMN NAMES
            # -----------------------------------------------------------------

            # Strip spaces from column names first
            forecast_df.columns = [str(c).strip() for c in forecast_df.columns]

            print("Original columns:")
            print(forecast_df.columns.tolist())

            # -------------------------------------------------------------
            # HANDLE DATE COLUMN
            # -------------------------------------------------------------

            if "Date" not in forecast_df.columns:

                print("Date column header missing - renaming first column to 'Date'")

                cols = list(forecast_df.columns)
                cols[0] = "Date"

                forecast_df.columns = cols

            # -------------------------------------------------------------
            # HANDLE ENTRIES COLUMN
            # -------------------------------------------------------------

            entries_col = None

            possible_entries_cols = [
                "Entries",
                "entry_forecast"
            ]

            for col in possible_entries_cols:
                if col in forecast_df.columns:
                    entries_col = col
                    break

            if entries_col is None:
                print("No entries forecast column found")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            # -------------------------------------------------------------
            # HANDLE EXITS COLUMN
            # -------------------------------------------------------------

            exits_col = None

            possible_exits_cols = [
                "Exits",
                "exit_forecast"
            ]

            for col in possible_exits_cols:
                if col in forecast_df.columns:
                    exits_col = col
                    break

            if exits_col is None:
                print("No exits forecast column found")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            print(f"Using entries column: {entries_col}")
            print(f"Using exits column:   {exits_col}")

            # -----------------------------------------------------------------
            # CLEAN DATE
            # -----------------------------------------------------------------
            date_sample = str(forecast_df["Date"].dropna().iloc[0])

            print(f"Sample date value: {date_sample}")

            # Detect format automatically
            if "-" in date_sample:
                
                # Likely YYYY-MM-DD
                forecast_df["Date"] = pd.to_datetime(
                    forecast_df["Date"],
                    format="%Y-%m-%d",
                    errors="coerce"
                )

            else:
                
                # Likely DD/MM/YYYY
                forecast_df["Date"] = pd.to_datetime(
                    forecast_df["Date"],
                    format="%d/%m/%Y",
                    errors="coerce"
                )

            # Debugging
            if forecast_df["Date"].isna().all():
                print("All forecast dates failed to parse")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue
            # -----------------------------------------------------------------
            # FILTER TO TARGET DATE
            # -----------------------------------------------------------------

            target_forecast = forecast_df[
                forecast_df["Date"].dt.date == target_date_only
            ]

            if target_forecast.empty:
                print("No matching forecast rows for target date")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            # -----------------------------------------------------------------
            # DAILY FORECAST TOTALS
            # -----------------------------------------------------------------

            forecast_entries = pd.to_numeric(
                target_forecast[entries_col],
                errors="coerce"
            ).fillna(0).sum()

            forecast_exits = pd.to_numeric(
                target_forecast[exits_col],
                errors="coerce"
            ).fillna(0).sum()

            print(f"Forecast Entries: {forecast_entries:.2f}")
            print(f"Forecast Exits:   {forecast_exits:.2f}")

            print(f"Actual Entries:   {actual_entries}")
            print(f"Actual Exits:     {actual_exits}")

            # Store forecasts
            entries_forecast_results[target_date_only][lead] = forecast_entries
            exits_forecast_results[target_date_only][lead] = forecast_exits

            # Store actuals
            entries_actual_results[target_date_only][lead] = actual_entries
            exits_actual_results[target_date_only][lead] = actual_exits


            # -----------------------------------------------------------------
            # ACCURACY CALC
            # Formula:
            # (Actual - Forecast) / Actual
            # -----------------------------------------------------------------

            if actual_entries != 0:
                entries_accuracy = 1-abs((
                    (actual_entries - forecast_entries)
                    / actual_entries
                ))
            else:
                entries_accuracy = np.nan

            if actual_exits != 0:
                exits_accuracy = 1-abs((
                    (actual_exits - forecast_exits)
                    / actual_exits
                ))
            else:
                exits_accuracy = np.nan

            entries_accuracy_results[target_date_only][lead] = entries_accuracy
            exits_accuracy_results[target_date_only][lead] = exits_accuracy

            print(f"Entries Accuracy: {entries_accuracy:.4f}")
            print(f"Exits Accuracy:   {exits_accuracy:.4f}")

        except Exception as e:

            print(f"ERROR reading forecast file:")
            print(f"{forecast_filename}")
            print(str(e))

            entries_accuracy_results[target_date_only][lead] = np.nan
            exits_accuracy_results[target_date_only][lead] = np.nan

# -----------------------------------------------------------------------------
# FINAL DATAFRAMES
# -----------------------------------------------------------------------------

entries_accuracy_df = pd.DataFrame.from_dict(
    entries_accuracy_results,
    orient="index"
)

exits_accuracy_df = pd.DataFrame.from_dict(
    exits_accuracy_results,
    orient="index"
)

entries_accuracy_df.index.name = "Date"
exits_accuracy_df.index.name = "Date"

entries_accuracy_df = entries_accuracy_df.sort_index()
exits_accuracy_df = exits_accuracy_df.sort_index()

lead_name_map = {
    42: "6_weeks_out",
    35: "5_weeks_out",
    28: "4_weeks_out",
    21: "3_weeks_out",
    14: "2_weeks_out",
    7: "1_week_out"
}

# Rename columns
entries_accuracy_df.columns = [
    lead_name_map[c]
    for c in entries_accuracy_df.columns
]

exits_accuracy_df.columns = [
    lead_name_map[c]
    for c in exits_accuracy_df.columns
]

# -----------------------------------------------------------------------------
# OUTPUT
# -----------------------------------------------------------------------------

print("\n================ ENTRIES ACCURACY ================\n")
print(entries_accuracy_df)

print("\n================ EXITS ACCURACY ================\n")
print(exits_accuracy_df)

# -----------------------------------------------------------------------------
# FORECAST VALUE TABLES
# -----------------------------------------------------------------------------

entries_forecast_df = pd.DataFrame.from_dict(
    entries_forecast_results,
    orient="index"
)

exits_forecast_df = pd.DataFrame.from_dict(
    exits_forecast_results,
    orient="index"
)

entries_forecast_df.index.name = "Date"
exits_forecast_df.index.name = "Date"

entries_forecast_df = entries_forecast_df.sort_index()
exits_forecast_df = exits_forecast_df.sort_index()

entries_forecast_df.columns = [
    lead_name_map[c]
    for c in entries_forecast_df.columns
]

exits_forecast_df.columns = [
    lead_name_map[c]
    for c in exits_forecast_df.columns
]

# -----------------------------------------------------------------------------
# ACTUAL VALUE TABLES
# -----------------------------------------------------------------------------

entries_actual_df = pd.DataFrame.from_dict(
    entries_actual_results,
    orient="index"
)

exits_actual_df = pd.DataFrame.from_dict(
    exits_actual_results,
    orient="index"
)

entries_actual_df.index.name = "Date"
exits_actual_df.index.name = "Date"

entries_actual_df = entries_actual_df.sort_index()
exits_actual_df = exits_actual_df.sort_index()

entries_actual_df.columns = [
    lead_name_map[c] for c in entries_actual_df.columns
]

exits_actual_df.columns = [
    lead_name_map[c] for c in exits_actual_df.columns
]

# -----------------------------------------------------------------------------
# PRINT OUTPUTS
# -----------------------------------------------------------------------------

print("\n================ ENTRIES FORECASTS ================\n")
print(entries_forecast_df)

print("\n================ EXITS FORECASTS ================\n")
print(exits_forecast_df)

print("\n================ ENTRIES ACTUALS ================\n")
print(entries_actual_df)

print("\n================ EXITS ACTUALS ================\n")
print(exits_actual_df)

# -----------------------------------------------------------------------------
# MONTHLY ACCURACY SUMMARY
# -----------------------------------------------------------------------------

entries_monthly = entries_accuracy_df.copy()
entries_monthly.index = pd.to_datetime(entries_monthly.index)

entries_monthly["Month"] = (
    entries_monthly.index.to_period("M")
)

monthly_entries_accuracy = (
    entries_monthly
    .groupby("Month")
    .mean()
)

print("\n================ MONTHLY ENTRIES ACCURACY ================\n")
print((monthly_entries_accuracy * 100).round(1))


exits_monthly = exits_accuracy_df.copy()
exits_monthly.index = pd.to_datetime(exits_monthly.index)

exits_monthly["Month"] = (
    exits_monthly.index.to_period("M")
)

monthly_exits_accuracy = (
    exits_monthly
    .groupby("Month")
    .mean()
)

print("\n================ MONTHLY EXITS ACCURACY ================\n")
print((monthly_exits_accuracy * 100).round(1))

Analysis period:
Start: 2026-03-29
End:   2026-10-20


C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_9628\4243284090.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fastpark_actual_entries_df = pd.read_sql(sql, connect)
C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_9628\4243284090.py:70: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fastpark_actual_exits_df = pd.read_sql(sql, connect)



Actuals daily:
        Date  ActualEntries  ActualExits
0 2026-03-29            407          434
1 2026-03-30            418          449
2 2026-03-31            338          363
3 2026-04-01            530          334
4 2026-04-02            596          292

Checking forecast file:
Target date: 2026-03-29
Lead time: 42
Forecast file: 2026-02-15_daily_forecast.csv
Loaded rows: 58
Original columns:
['Unnamed: 0', 'booking_entries', 'booking_exits', 'entry_forecast', 'exit_forecast']
Date column header missing - renaming first column to 'Date'
Using entries column: entry_forecast
Using exits column:   exit_forecast
Sample date value: 2026-02-16
Forecast Entries: 493.00
Forecast Exits:   564.00
Actual Entries:   407
Actual Exits:     434
Entries Accuracy: 0.7887
Exits Accuracy:   0.7005

Checking forecast file:
Target date: 2026-03-29
Lead time: 35
Forecast file: 2026-02-22_daily_forecast.csv
Loaded rows: 58
Original columns:
['Unnamed: 0', 'booking_entries', 'booking_exits', 'entry_fo

save to excel

In [16]:
# -----------------------------------------------------------------------------
# SAVE TO EXCEL
# -----------------------------------------------------------------------------

output_folder = Path(
    rf"C:\Users\{user}\OneDrive - Edinburgh Airport Limited\Documents\Fast Park Report\Summer Wash-up 2026"
)

output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / (
    f"forecast_accuracy_{pd.Timestamp.today().strftime('%d%m%y')}.xlsx"
)

# -----------------------------------------------------------------------------
# WRITE TO EXCEL
# -----------------------------------------------------------------------------

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    sheet_name = "Forecast Accuracy"

    # -------------------------------------------------------------------------
    # ENTRIES SECTION
    # -------------------------------------------------------------------------

    current_row = 0

    entries_forecast_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet = writer.sheets[sheet_name]

    worksheet.cell(
        row=current_row + 1,
        column=1
    ).value = "Entries Forecast"

    current_row += len(entries_forecast_df) + 4

    entries_actual_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet.cell(
        row=current_row + 1,
        column=1
    ).value = "Entries Actuals"

    current_row += len(entries_actual_df) + 4

    entries_accuracy_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet.cell(
        row=current_row + 1,
        column=1
    ).value = "Entries Accuracy"

    # -------------------------------------------------------------------------
    # EXITS SECTION
    # -------------------------------------------------------------------------

    exits_start_col = len(entries_forecast_df.columns) + 4

    current_row = 0

    exits_forecast_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Forecast"

    current_row += len(exits_forecast_df) + 4

    exits_actual_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Actuals"

    current_row += len(exits_actual_df) + 4

    exits_accuracy_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Accuracy"

    # -------------------------------------------------------------------------
    # MONTHLY SUMMARY SHEETS
    # -------------------------------------------------------------------------

    monthly_entries_accuracy.to_excel(
        writer,
        sheet_name="Monthly Entries Accuracy"
    )

    monthly_exits_accuracy.to_excel(
        writer,
        sheet_name="Monthly Exits Accuracy"
    )

    monthly_entry_ws = writer.sheets["Monthly Entries Accuracy"]
    monthly_exit_ws = writer.sheets["Monthly Exits Accuracy"]

    # -------------------------------------------------------------------------
    # FORMATTING
    # -------------------------------------------------------------------------

    from openpyxl.styles import Font

    bold_font = Font(bold=True)

    # Main sheet titles
    for row in worksheet.iter_rows():
        for cell in row:
            if cell.value in [
                "Entries Forecast",
                "Entries Actuals",
                "Entries Accuracy",
                "Exits Forecast",
                "Exits Actuals",
                "Exits Accuracy"
            ]:
                cell.font = bold_font

    # Monthly sheet titles
    monthly_entry_ws["A1"].font = bold_font
    monthly_exit_ws["A1"].font = bold_font

    # Monthly sheets as %
    for ws in [monthly_entry_ws, monthly_exit_ws]:

        for row in ws.iter_rows(
            min_row=2,
            min_col=2
        ):
            for cell in row:

                if isinstance(cell.value, (int, float)):
                    cell.number_format = "0.0%"

    # Auto-width all sheets
    for ws in [worksheet, monthly_entry_ws, monthly_exit_ws]:

        for column_cells in ws.columns:

            length = 0
            column = column_cells[0].column_letter

            for cell in column_cells:

                try:
                    length = max(
                        length,
                        len(str(cell.value))
                    )
                except:
                    pass

            ws.column_dimensions[column].width = length + 3

print(f"\nExcel saved:")
print(output_file)


Excel saved:
C:\Users\jamie_douglas\OneDrive - Edinburgh Airport Limited\Documents\Fast Park Report\Summer Wash-up 2026\forecast_accuracy_211026.xlsx


### Entries Monthly Accuracy Trend

In [17]:
import plotly.graph_objects as go

fig = go.Figure()

for col in monthly_entries_accuracy.columns:

    fig.add_trace(
        go.Scatter(
            x=monthly_entries_accuracy.index.astype(str),
            y=monthly_entries_accuracy[col] * 100,
            mode="lines+markers",
            name=col.replace("_", " ").title()
        )
    )

fig.update_layout(
    title="FastPark Entries Forecast Accuracy by Arrival Month",
    xaxis_title="Arrival Month",
    yaxis_title="Accuracy (%)",
    yaxis=dict(
        ticksuffix="%",
        range=[50, 100]
    ),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

### Exits Monthly Accuracy Trend

In [18]:
import plotly.graph_objects as go

fig = go.Figure()

for col in monthly_exits_accuracy.columns:

    fig.add_trace(
        go.Scatter(
            x=monthly_exits_accuracy.index.astype(str),
            y=monthly_exits_accuracy[col] * 100,
            mode="lines+markers",
            name=col.replace("_", " ").title()
        )
    )

fig.update_layout(
    title="FastPark Exits Forecast Accuracy by Arrival Month",
    xaxis_title="Arrival Month",
    yaxis_title="Accuracy (%)",
    yaxis=dict(
        ticksuffix="%",
        range=[50, 100]
    ),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

### Entries Heatmap

In [19]:
import plotly.express as px

heatmap_df = monthly_entries_accuracy.copy()
heatmap_df.index = heatmap_df.index.astype(str)

fig = px.imshow(
    heatmap_df * 100,
    text_auto=".0f",
    aspect="auto",
    color_continuous_scale=[
        [0.00, "#F2B8A0"],  # soft coral (<70)
        [0.70, "#F2B8A0"],

        [0.70, "#E8D7A7"],  # warm sand (70-80)
        [0.80, "#E8D7A7"],

        [0.80, "#C9DEAE"],  # sage green (80-90)
        [0.90, "#C9DEAE"],

        [0.90, "#77C66E"],  # green (90-95)
        [0.95, "#77C66E"],

        [0.95, "#2E8B57"],  # dark green (95-100)
        [1.00, "#2E8B57"]
    ],
    zmin=60,
    zmax=100,
    labels=dict(
        x="Forecast Version",
        y="Arrival Month",
        color="Accuracy %"
    )
)

fig.update_layout(
    title="FastPark Entries Forecast Accuracy"
)

fig.show()

### Exits Heatmap

In [20]:
import plotly.express as px

heatmap_df = monthly_exits_accuracy.copy()
heatmap_df.index = heatmap_df.index.astype(str)

fig = px.imshow(
    heatmap_df * 100,
    text_auto=".0f",
    aspect="auto",
    color_continuous_scale=[
        [0.00, "#d7191c"],
        [0.70, "#d7191c"],

        [0.70, "#f46d43"],
        [0.75, "#f46d43"],

        [0.75, "#fdae61"],
        [0.80, "#fdae61"],

        [0.80, "#ffffbf"],
        [0.85, "#ffffbf"],

        [0.85, "#d9ef8b"],
        [0.90, "#d9ef8b"],

        [0.90, "#66bd63"],
        [0.95, "#66bd63"],

        [0.95, "#1a9641"],
        [1.00, "#1a9641"]
    ],
    zmin=0,
    zmax=100,
    labels=dict(
        x="Forecast Version",
        y="Arrival Month",
        color="Accuracy %"
    )
)

fig.update_layout(
    title="FastPark Exits Forecast Accuracy Heatmap"
)

fig.show()

### Average Accuracy by Forecast Version (Entries)

In [22]:
import plotly.graph_objects as go

average_entries_accuracy = (
    monthly_entries_accuracy.mean() * 100
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=[
            c.replace("_", " ").title()
            for c in average_entries_accuracy.index
        ],
        y=average_entries_accuracy.values,
        text=average_entries_accuracy.round(1),
        textposition="outside"
    )
)

fig.update_layout(
    title="Average Entries Forecast Accuracy by Forecast Version",
    xaxis_title="Forecast Version",
    yaxis_title="Accuracy (%)",
    yaxis=dict(
        range=[0, 100]
    ),
    template="plotly_white"
)

fig.show()

### Average Accuracy by Forecast Version (Exits)

In [23]:
import plotly.graph_objects as go

average_exits_accuracy = (
    monthly_exits_accuracy.mean() * 100
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=[
            c.replace("_", " ").title()
            for c in average_exits_accuracy.index
        ],
        y=average_exits_accuracy.values,
        text=average_exits_accuracy.round(1),
        textposition="outside"
    )
)

fig.update_layout(
    title="Average Exits Forecast Accuracy by Forecast Version",
    xaxis_title="Forecast Version",
    yaxis_title="Accuracy (%)",
    yaxis=dict(
        range=[0, 100]
    ),
    template="plotly_white"
)

fig.show()

In [24]:
import pandas as pd
import plotly.graph_objects as go

entries_avg = monthly_entries_accuracy.mean() * 100
exits_avg = monthly_exits_accuracy.mean() * 100

fig = go.Figure()

fig.add_trace(
    go.Bar(
        name="Entries",
        x=entries_avg.index,
        y=entries_avg.values
    )
)

fig.add_trace(
    go.Bar(
        name="Exits",
        x=exits_avg.index,
        y=exits_avg.values
    )
)

fig.update_layout(
    title="Average Forecast Accuracy by Forecast Version",
    xaxis_title="Forecast Version",
    yaxis_title="Accuracy (%)",
    yaxis=dict(range=[0,100]),
    barmode="group",
    template="plotly_white"
)

fig.show()

In [25]:
entries_avg = monthly_entries_accuracy.mean() * 100

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=entries_avg.index,
        y=entries_avg.values,
        mode="lines+markers",
        line=dict(width=4),
        marker=dict(size=10),
        name="Entries"
    )
)

fig.update_layout(
    title="Forecast Accuracy Improvement Curve",
    xaxis_title="Forecast Version",
    yaxis_title="Accuracy (%)",
    yaxis=dict(range=[60,100]),
    template="plotly_white"
)

fig.show()

In [26]:
import plotly.graph_objects as go

fig = go.Figure()

for col in monthly_entries_accuracy.columns:

    fig.add_trace(
        go.Box(
            y=entries_accuracy_df[col] * 100,
            name=col
        )
    )

fig.update_layout(
    title="Distribution of Forecast Accuracy",
    yaxis_title="Accuracy (%)",
    template="plotly_white"
)

fig.show()